# Safety-circuits — §9 editing suite (all-in-one runner)

Runs the **full editing study + every Tier 1/2 extension for one model in a single pass**.
Change only `MODEL`, then **Save Version → Save & Run All (Commit)**; download
`safety_circuits_editing_results.zip` from the version's Output and commit `editing/<MODEL>/`
under `results/editing/<MODEL>/`.

**One-time notebook settings:** Accelerator = **GPU T4**, Internet = **On**, and add an
`HF_TOKEN` under Add-ons → Secrets (HF account must have accepted the gemma-3 / llama-3.2 gated terms).

**Runtime:** with everything on this trains ~12 LoRA adapters + long-gen evals → roughly **3–4 h/model**
(within the 12 h cap). To go faster: set `SC_DO_MINIMAL_SWEEP=0` or shrink its grid; `SC_DO_HARDENING=0`.
First time, smoke-test with `SC_EDIT_STEPS=50` to confirm all CSVs write.

**Ethics:** the benign substance-unlock test (T1.1b) trains/evaluates on **benign content only**;
we do not train on or elicit harmful generation (see `RESEARCH_PLAN.md` §9 ethics).

Outputs in `editing/<MODEL>/`: `*_edit_summary.csv`, `*_scalpel_axis.png`, `*_edit_steering_sweep.csv`,
`*_edit_headcount_sweep.*`, `*_edit_generalization.csv`, `*_edit_direction_shift.csv`,
`*_edit_roundtrip.csv`, `*_edit_minimal_sweep.csv`, `*_edit_benign_substance.csv` (+`_agg`), `_EDIT_DONE.json`.

In [ ]:
import os, pathlib, subprocess, sys

MODEL = "gemma3-1b"   # <-- the only knob. one of:
                      # gemma3-1b, qwen3, qwen2.5, qwen2-1.5b, qwen1.5-1.8b,
                      # gemma1-2b, gemma2-2b, llama3.2-1b, llama3-3b

# ---- core editing run (validated config) ----
os.environ["SC_MODELS"]               = MODEL
os.environ["SC_EDIT_STEPS"]           = "600"
os.environ["SC_EDIT_RANK"]            = "16"
os.environ["SC_EDIT_LR"]              = "5e-4"
os.environ["SC_EDIT_HEADCOUNTS"]      = "1,3,5,10"
os.environ["SC_EDIT_METHODS"]         = "steering,lora"

# ---- Tier 1/2 extensions: everything ON ----
os.environ["SC_DO_GENERALIZATION"]    = "1"        # long-form + per-category + toxicity        (T1.1)
os.environ["SC_DO_DIRSHIFT"]          = "1"        # refusal-direction rotation (mechanism)      (T2.6/2.7)
os.environ["SC_DO_HARDENING"]         = "1"        # comply->refuse safety re-patch round-trip   (T2.5)
os.environ["SC_DO_MINIMAL_SWEEP"]     = "1"        # smallest clean edit                         (T1.2)
os.environ["SC_EDIT_MINIMAL_RANKS"]   = "8,16"     # trim the sweep grid (2x2 = 4 trains)
os.environ["SC_EDIT_MINIMAL_STEPS"]   = "300,600"
os.environ["SC_DO_BENIGN_SUBSTANCE"]  = "1"        # weapon-free substance-unlock test           (T1.1b)
os.environ["SC_EDIT_MAX_TARGET_TOKENS"] = "128"    # long benign target length

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- HF token for gated models (gemma, llama) ----
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

# ---- deps + drop ABI-mismatched torchaudio/torchvision (unused; would crash the import) ----
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformer-lens>=2.0.0", "transformers>=4.40", "datasets>=2.18",
    "accelerate>=0.27", "einops>=0.7", "seaborn", "tqdm", "pyyaml", "jaxtyping",
    "peft>=0.10"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio", "torchvision"],
               check=False)

# ---- pull latest repo ----
repo = pathlib.Path("/tmp/safety-circuits")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/pra-nav-04/safety-circuits.git", str(repo)], check=True)
sys.path.insert(0, str(repo / "src"))

# ---- purge stale cached modules so the freshly pulled code loads on a warm kernel ----
for _m in [m for m in list(sys.modules)
           if m.split(".")[0] in ("safety_circuits", "transformer_lens", "transformers",
                                   "torchaudio", "torchvision")]:
    del sys.modules[_m]

import runpy
runpy.run_path(str(repo / "kaggle" / "run_edit_experiment.py"), run_name="__main__")